# 12 · Interseismic coupling

Every previous chapter modeled an *earthquake*: a fault slipping. This chapter
models the quiet decades *between* earthquakes, when a fault is locked and
strain is slowly building. The measurements are steady GNSS velocities rather
than sudden displacements, but with one clever trick, the **backslip** model,
the whole problem becomes the same linear inversion we already know. The
payoff is a map of how strongly the fault is locked, which is directly relevant
to where future earthquakes may nucleate.

**Learning objectives**

By the end of the chapter, you will be able to:

- read the arctangent surface-velocity profile of a locked fault
- explain the backslip idea and GeoDef's sign convention for it
- estimate bounded plate-parallel backslip from surface velocities
- convert backslip rate into coupling and a moment-deficit rate

**Prerequisites:** Chapters 05 and 07
**Estimated time:** 45–60 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. What the surface does between earthquakes

Two plates converge at a steady rate, but the fault between them is stuck, or
**locked**, in its shallow part. Because the deep plate keeps moving while the
shallow fault does not, elastic strain accumulates in the crust around the
locked zone. GNSS stations record this as a steady velocity that varies
smoothly with distance from the fault. When the fault finally breaks, that
stored strain is released as an earthquake.

Our goal is to invert those steady velocities for the pattern of locking: which
parts of the fault are fully stuck (and storing strain for a future
earthquake), and which are creeping freely (and storing none).

## 2. The arctangent profile

The simplest locked fault, an infinitely long vertical strike-slip fault stuck
from the surface down to a locking depth $D$ and moving at far-field relative
rate $V$, produces a surface velocity that follows an arctangent:

$$ v(x) = \frac{V}{\pi}\arctan\left(\frac{x}{D}\right). $$

Here $x$ is the distance from the fault. The shape is worth internalizing: the
velocity transitions smoothly across the fault, and the *width* of that
transition is set by the locking depth. A shallow lock gives a sharp step; a
deep lock gives a broad, gentle ramp. This is why the near-fault velocity
gradient, not the far-field rate, is what pins down how deep the fault is
locked.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

In [ ]:
x_km = np.linspace(-100.0, 100.0, 401)
plate_rate_mm_yr = 40.0
fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
for locking_depth_km in (5.0, 15.0, 30.0):
    velocity = plate_rate_mm_yr / np.pi * np.arctan(x_km / locking_depth_km)
    ax.plot(x_km, velocity, label=f"locking depth {locking_depth_km:g} km")
ax.set(xlabel="distance from fault (km)",
       ylabel="fault-parallel velocity (mm/yr)",
       title="Surface velocity over a locked strike-slip fault")
ax.legend()
plt.show()

## 3. The backslip trick

How do we turn "the fault is locked" into a slip inversion? The **backslip**
model (Savage, 1983) is the key. Imagine two pieces added together:

1. the two plates sliding past each other *everywhere*, including on the fault,
   at the full plate rate; plus
2. an equal and *opposite* slip, only on the locked part of the fault.

The steady sliding produces no surface deformation (rigid blocks just
translate), so all the observed strain comes from piece 2: the opposite-sense
slip on the locked patches. That opposite slip is the **backslip**. It is a
purely kinematic bookkeeping device, not a real motion and not a friction law,
but it lets us image locking with the very same dislocation machinery used for
earthquakes.

## 4. The sign convention

Because backslip is anti-parallel to plate motion, its sign takes care on a
thrust fault. Positive backslip amplitude points *against* the convergence
direction, which on a megathrust is normal-sense motion, so its raw dip-slip
component comes out negative. Rather than track that minus sign by hand, we use
the **plate basis**: we tell GeoDef the plate-motion direction with
`plate_rake`, and it solves for a backslip amplitude that we bound between zero
and the local plate rate. The binding sign rules are recorded in
[the conventions guide](../docs/conventions.md#interseismic-backslip-and-coupling).

## 5. Build a small coupling scenario

We set up a thrust fault and prescribe a patch of strong locking near its
center. The synthetic velocities we generate are assumed to be already
corrected into a frame fixed to the overriding plate (Section 7 explains why
that correction matters). The convergence rate is 50 mm/yr.

In [ ]:
rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=12_000.0, strike=90.0, dip=20.0,
    length=40_000.0, width=24_000.0, n_length=5, n_width=3,
)
plate_rate = 0.05        # 50 mm/yr convergence, in m/yr
plate_rake = -90.0       # normal-sense backslip direction on this thrust

**Coupling** is the locked fraction: 1 where the fault is fully stuck, 0 where
it creeps freely. We prescribe a smooth patch of high coupling and turn it into
a backslip rate by multiplying by the plate rate, then split that into strike
and dip components with the plate rake.

In [ ]:
along = np.arange(fault.n_patches) % 5
down = np.arange(fault.n_patches) // 5
coupling_true = np.exp(-((along - 2.0) / 1.6) ** 2 - ((down - 0.7) / 1.0) ** 2)
backslip_true = plate_rate * coupling_true      # locked fraction -> backslip rate
strike_rate, dip_rate = geodef.slip.from_rake(backslip_true, rake_degrees=plate_rake)

Forward-model the backslip to a GNSS network and add 0.8 mm/yr velocity noise.
The dataset is tagged as a velocity in m/yr so its physical meaning stays
explicit.

In [ ]:
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.30, -117.70, 8), np.linspace(33.82, 34.22, 6)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=strike_rate, slip_dip=dip_rate
)
sigma = 0.0008           # 0.8 mm/yr
velocity = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    quantity="velocity", units="m/yr", name="block_corrected_gnss",
)

## 6. Invert for coupling

The solve uses the plate basis. We fix the plate rake, smooth lightly, and
bound the amplitude: the plate-parallel component between 0 and the plate rate
(coupling cannot be negative or exceed full locking), and the perpendicular
component left free. The two-element bound arrays set those limits for the two
plate-basis components.

In [ ]:
result = geodef.solve(
    fault, datasets=velocity, components="plate", plate_rake=plate_rake,
    regularization="laplacian", regularization_strength=1.0,
    # bounds on [parallel, perpendicular]: parallel in [0, plate_rate].
    bounds=(np.array([0.0, -np.inf]), np.array([plate_rate, np.inf])),
)

The plate-parallel solution is a backslip *rate*. Dividing it by the local
plate rate turns it back into a coupling fraction between 0 and 1. The
**moment-deficit rate** is the rate at which locked patches accumulate the
equivalent of seismic moment, a running tally of the earthquake potential being
stored.

In [ ]:
coupling = result.rake_parallel / plate_rate
moment_deficit_rate = fault.medium.mu * np.sum(fault.areas * result.rake_parallel)
print(f"moment-deficit rate: {moment_deficit_rate:.3e} N m / yr")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
geodef.plot.patches(fault, coupling_true, ax=axes[0], vmin=0, vmax=1,
                    title="Input coupling", colorbar_label="Coupling")
geodef.plot.patches(fault, coupling, ax=axes[1], vmin=0, vmax=1,
                    title="Recovered coupling", colorbar_label="Coupling")
plt.show()

The locked patch is recovered. Two honesty notes. First, the formal
uncertainty on this result still excludes the larger error sources: Euler-pole
error in the frame correction, block-model error, and any viscoelastic or
transient deformation the elastic model ignores (Chapter 13). Second, a
moment-deficit rate is a rate of *accumulation*, not an earthquake forecast; it
says nothing about when or how the stored moment will be released.

**Checkpoint.** The moment-deficit rate summed `fault.areas * result.rake_parallel`.
Why weight each patch's backslip by its area before summing?

<details><summary>Answer</summary>

Moment is force times distance integrated over the slipping surface, so a large
patch storing a given backslip rate contributes more than a small patch storing
the same rate. Multiplying by area (and by the shear modulus) converts a
per-patch slip rate into the physical moment rate, exactly as the coseismic
moment in Chapter 01 weighted slip by patch area.

</details>

## 7. Why the reference frame matters

We assumed the velocities were already in an overriding-plate-fixed frame. That
step is not optional. Raw GNSS velocities include the rigid rotation of the
whole block the stations sit on. If you invert those directly, that rigid
motion masquerades as elastic strain and corrupts the coupling estimate. The
correction removes the block's rigid rotation (using an Euler pole; see the
[Euler-pole documentation](../docs/euler.md)) so that what remains is the
elastic loading you actually want to invert. The uncertainty in that pole then
becomes part of the data uncertainty.

## Checkpoint questions

**Why correct the reference frame before a coupling inversion?**

<details><summary>Answer</summary>

Raw velocities include rigid block motion. Left in, that motion looks like
elastic strain or locking and biases the coupling estimate. Correcting into an
overriding-plate-fixed frame leaves only the elastic loading the model is
meant to explain.

</details>

**Why is a positive backslip amplitude a negative raw dip-slip here?**

<details><summary>Answer</summary>

Backslip is anti-parallel to plate motion. On a thrust (convergent) fault, the
direction opposite to convergence is normal-sense, so the raw dip-slip
component of positive backslip is negative. The plate basis hides this
bookkeeping behind a single non-negative amplitude.

</details>

**Is backslip a physical friction law?**

<details><summary>Answer</summary>

No. It is a kinematic superposition trick that reproduces the surface strain of
a locked fault using ordinary dislocations. It makes no claim about the friction
or physics on the fault itself.

</details>

## Common mistakes

- **Inverting uncorrected velocities.** Mixing block rotation into the data
  confuses rigid motion with elastic loading and biases coupling.
- **Dividing by one scalar rate on a curved boundary.** Coupling is backslip
  relative to the *local* plate rate; using a single rate where convergence
  varies misstates the fraction.
- **Plotting signed dip slip instead of coupling.** The physically meaningful
  quantity is the locked fraction between 0 and 1, not the raw signed
  component.
- **Reading the moment-deficit rate as a forecast.** It measures accumulation,
  not release timing.

## Recap

- Between earthquakes, a locked fault produces a smooth arctangent velocity
  profile whose width reflects the locking depth.
- Backslip represents locking as opposite-sense slip, turning coupling into the
  familiar linear inversion.
- The plate basis and non-negative bounds keep the sign convention and the
  coupling range physical.
- Coupling is bounded backslip divided by the local plate rate; the
  moment-deficit rate accumulates stored earthquake potential.

Chapter 13 confronts the error sources these formal uncertainties leave out,
including the frame and elastic-model assumptions used here.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/12_interseismic_coupling_solutions.ipynb`.

1. **Locking depth.** Fit arctangent profiles for several locking depths and
   confirm how the transition width tracks the depth.
2. **Down-dip recoverability.** Move the coupled patch deeper on the fault and
   compare how well it is recovered.
3. **Plate-rate bias.** Perturb the assumed plate rate by 10 percent and
   quantify the resulting bias in coupling and moment-deficit rate.
4. **Challenge: raw coordinates.** Rebuild the synthetic problem in raw
   dip-slip coordinates (carrying the minus sign yourself) and verify it
   reproduces the plate-basis prediction exactly.

## Further reading

- Savage, J. C. (1983), "A dislocation model of strain accumulation and release
  at a subduction zone," *Journal of Geophysical Research*, 88(B6), 4984–4996,
  the backslip model.
- Segall, P. (2010), *Earthquake and Volcano Deformation*, on earthquake-cycle
  geodesy and coupling.
- McCaffrey, R. (2002), on block models and GPS velocities.
- [GeoDef Euler-pole documentation](../docs/euler.md) for the frame correction.
- The next course chapter is `13_model_misspecification.ipynb`.